In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Environment Setup ===
Conda env requirements: Python 3.11+, scikit-image, Pillow, ipykernel, JupyterLab, tqdm; register the env as a Jupyter kernel with ipykernel.

E.g.: 
conda create -n scikit-image python=3.11
conda activate scikit-image
conda install -c conda-forge scikit-image pillow ipykernel jupyterlab tqdm
python -m ipykernel install --user --name scikit-image --display-name "scikit-image"

In [ ]:
# === Cell 3: Resize Images to 480/720/1080p ===
from pathlib import Path
import csv
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

input_dir = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Update_Extended_Image_Dataset")
out_480 = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/480_resolution")
out_720 = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/720_resolution")
out_1080 = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/1080_resolution")

for out_dir in [out_480, out_720, out_1080]:
    out_dir.mkdir(parents=True, exist_ok=True)

# Target image height (px) -> output folder; width keeps the aspect ratio.
target_heights = {
    480: out_480,
    720: out_720,
    1080: out_1080
}

valid_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}

image_files = [
    p for p in input_dir.iterdir()
    if p.is_file() and p.suffix.lower() in valid_exts
]

# Unreadable files are collected and written to a CSV report.
processed = 0
skipped = []

for img_path in tqdm(image_files, desc="Processing images", unit="image"):
    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            original_width, original_height = img.size

            for target_h, out_dir in target_heights.items():
                target_w = round(original_width * target_h / original_height)
                resized = img.resize((target_w, target_h), Image.Resampling.LANCZOS)

                save_path = out_dir / img_path.name
                if save_path.suffix.lower() in {".jpg", ".jpeg"}:
                    resized.save(save_path, quality=95)
                else:
                    resized.save(save_path)

        processed += 1

    except (UnidentifiedImageError, OSError, SyntaxError, ValueError) as e:
        skipped.append({
            "filename": img_path.name,
            "full_path": str(img_path),
            "extension": img_path.suffix.lower(),
            "error": str(e)
        })

if skipped:
    report_path = input_dir / "skipped_files_report.csv"
    with open(report_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["filename", "full_path", "extension", "error"]
        )
        writer.writeheader()
        writer.writerows(skipped)

print(f"Processed images: {processed}")
print(f"Skipped files: {len(skipped)}")
print(f"480p output folder:        {out_480}")
print(f"720p output folder:        {out_720}")
print(f"1080p output folder:       {out_1080}")

if skipped:
    print(f"Skip report written to:    {report_path}")
    print("\nFirst few skipped files:")
    for row in skipped[:10]:
        print(f"- {row['filename']} | {row['error']}")
else:
    print("No skipped files!")